<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS02_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS02 — Praproses Data (Preprocessing) pada Dataset RumahPoint

Jobsheet asli memakai Titanic dan dataset kanker payudara (WBC). Versi PBL ini
menerapkan langkah yang sama pada data properti: membersihkan duplikat/outlier,
menangani nilai hilang, mengubah tipe data, menurunkan fitur, melakukan encoding,
dan standardisasi.

**Alur:** muat data → hapus duplikat & outlier → perbaiki `usia_bangunan` → turunkan
`kondisi` dan `kelas_harga` → imputasi → encoding `lokasi` → standardisasi.

## Langkah 1 — Muat Data Mentah

Dataset yang dipakai: `dataset/properti_multi_kota.csv` — gabungan beberapa portal
properti nasional (64.650 baris).

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = 'dataset/properti_multi_kota.csv'
TAHUN_INI = 2026
df = pd.read_csv(DATA_PATH)

print(f'Shape awal: {df.shape}')
df.head()

Shape awal: (64650, 8)


,sumber,lokasi,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,usia_bangunan,harga
0,uas-properti/Jabodetabek,Tangerang,30.0,NaN,3.0,3.0,33.0,2.037543e+08
1,uas-properti/Jabodetabek,Bekasi,122.0,NaN,5.0,4.0,19.0,1.058032e+09
2,uas-properti/Jabodetabek,Bekasi,142.8,NaN,1.0,1.0,9.0,1.335793e+09
3,uas-properti/Jabodetabek,Bekasi,152.6,NaN,4.0,3.0,6.0,1.613771e+09
4,uas-properti/Jabodetabek,Depok,61.2,NaN,1.0,1.0,9.0,5.221562e+08


## Langkah 2 — Hapus Duplikat dan Nilai Tidak Wajar

Data hasil scraping sering mengandung baris ganda dan kesalahan entri (luas 0 m²,
harga puluhan triliun, kamar 99). Aturan yang dipakai sama dengan laporan UTS agar
hasil antar-jobsheet konsisten.

In [2]:
catatan = []

before = len(df)
df = df.drop_duplicates()
catatan.append(('Hapus duplikat', before, len(df)))

before = len(df)
df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
df = df[df['luas_bangunan'].isna() |
        ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
df = df[df['harga'].between(50_000_000, 100_000_000_000)]
df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]
catatan.append(('Hapus outlier', before, len(df)))

for nama, awal, akhir in catatan:
    print(f'{nama:20}: {awal:>7,} -> {akhir:>7,}  (dibuang {awal - akhir:,})')
print(f'\nShape setelah pembersihan: {df.shape}')

Hapus duplikat      :  64,650 ->  57,494  (dibuang 7,156)
Hapus outlier       :  57,494 ->  56,170  (dibuang 1,324)

Shape setelah pembersihan: (56170, 8)


## Langkah 3 — Perbaiki `usia_bangunan`

Sebagian sumber menyimpan **tahun bangunan** (mis. `2020`) alih-alih usia. Nilai di atas
100 dianggap tahun dan dikonversi menjadi `TAHUN_INI - nilai`. Nilai yang masih di luar
rentang 0–100 dianggap tidak valid lalu dijadikan `NaN`.

Setelah itu diturunkan fitur **`kondisi`** dengan ambang **3 tahun**:

- `Baru` → usia ≤ 3 tahun
- `Bekas` → usia > 3 tahun
- `Tidak diketahui` → usia kosong

In [3]:
tahun = df['usia_bangunan'] > 100
df.loc[tahun, 'usia_bangunan'] = TAHUN_INI - df.loc[tahun, 'usia_bangunan']
df.loc[~df['usia_bangunan'].between(0, 100) & df['usia_bangunan'].notna(),
       'usia_bangunan'] = np.nan

df['kondisi'] = np.where(
    df['usia_bangunan'].isna(), 'Tidak diketahui',
    np.where(df['usia_bangunan'] <= 3, 'Baru', 'Bekas'))

print(f'Jumlah nilai tahun yang dikonversi: {tahun.sum():,}')
print('\nDistribusi kondisi:')
print(df['kondisi'].value_counts().to_string())
print(f'\nPersentase usia_bangunan yang terisi: '
      f'{df["usia_bangunan"].notna().mean() * 100:.1f}%')

Jumlah nilai tahun yang dikonversi: 2,757

Distribusi kondisi:
kondisi
Tidak diketahui    51838
Bekas               3415
Baru                 917

Persentase usia_bangunan yang terisi: 7.7%


## Langkah 4 — Bentuk Label Klasifikasi

Harga dipotong menjadi tiga kelas berimbang memakai kuantil 1/3 dan 2/3. Pendekatan
ini sama dengan `klasifikasi/Preprocessing.ipynb` supaya label antar-notebook konsisten.

In [4]:
q1, q2 = df['harga'].quantile([1 / 3, 2 / 3])

df['kelas_harga'] = np.where(
    df['harga'] < q1, 'Murah',
    np.where(df['harga'] <= q2, 'Sedang', 'Mahal'))

print(f'Batas Murah/Sedang : Rp {q1:,.0f}')
print(f'Batas Sedang/Mahal : Rp {q2:,.0f}')
print()
print(df['kelas_harga'].value_counts().reindex(['Murah', 'Sedang', 'Mahal']).to_string())

df[['lokasi', 'luas_m2', 'luas_bangunan', 'jumlah_kamar',
    'jumlah_kamar_mandi', 'usia_bangunan', 'kondisi', 'harga', 'kelas_harga']].head()

Batas Murah/Sedang : Rp 483,819,314
Batas Sedang/Mahal : Rp 1,235,000,000

kelas_harga
Murah     18723
Sedang    18727
Mahal     18720


,lokasi,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,usia_bangunan,kondisi,harga,kelas_harga
0,Tangerang,30.0,NaN,3.0,3.0,33.0,Bekas,2.037543e+08,Murah
1,Bekasi,122.0,NaN,5.0,4.0,19.0,Bekas,1.058032e+09,Sedang
2,Bekasi,142.8,NaN,1.0,1.0,9.0,Bekas,1.335793e+09,Mahal
3,Bekasi,152.6,NaN,4.0,3.0,6.0,Bekas,1.613771e+09,Mahal
4,Depok,61.2,NaN,1.0,1.0,9.0,Bekas,5.221562e+08,Sedang


## Langkah 5 — Tangani Nilai Hilang

`luas_bangunan`, `jumlah_kamar`, dan `jumlah_kamar_mandi` masih memiliki nilai kosong
dalam jumlah kecil. Karena distribusinya miring, imputasi memakai **median** (bukan
rata-rata) agar tidak tertarik oleh nilai ekstrem.

In [5]:
FITUR_NUM = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']

print('Nilai hilang sebelum imputasi:')
print(df[FITUR_NUM].isna().sum().to_string())

median = df[FITUR_NUM].median()
df[FITUR_NUM] = df[FITUR_NUM].fillna(median)

print('\nMedian yang dipakai:')
print(median.round(2).to_string())
print('\nNilai hilang setelah imputasi:', int(df[FITUR_NUM].isna().sum().sum()))

Nilai hilang sebelum imputasi:
luas_m2                  0
luas_bangunan         1531
jumlah_kamar           342
jumlah_kamar_mandi     353

Median yang dipakai:
luas_m2               100.0
luas_bangunan          65.0
jumlah_kamar            2.0
jumlah_kamar_mandi      2.0

Nilai hilang setelah imputasi: 0


## Langkah 6 — Encoding `lokasi`

`lokasi` memiliki ratusan ejaan (mis. "Bekasi", "Kabupaten Bekasi", "BEKASI"). Untuk
model tabular, dua teknik yang paling praktis:

1. **Frequency encoding** — ganti nama lokasi dengan proporsi kemunculannya. Murah secara
   komputasi dan tidak menambah dimensi.
2. **One-hot encoding** untuk 10 lokasi terbanyak + kategori `Lainnya`.

Keduanya didemonstrasikan; pemilihan final dilakukan di JS03.

In [6]:
df['lokasi_bersih'] = df['lokasi'].str.strip().str.title()
print(f'Ejaan lokasi unik: {df["lokasi"].nunique():,} -> '
      f'setelah title-case: {df["lokasi_bersih"].nunique():,}')

frekuensi = df['lokasi_bersih'].value_counts(normalize=True)
df['lokasi_freq'] = df['lokasi_bersih'].map(frekuensi)

top10 = frekuensi.head(10).index
df['lokasi_top'] = np.where(df['lokasi_bersih'].isin(top10),
                            df['lokasi_bersih'], 'Lainnya')

ohe = pd.get_dummies(df['lokasi_top'], prefix='lok', dtype=int)
print(f'Dimensi one-hot (10 lokasi + Lainnya): {ohe.shape[1]} kolom')
ohe.head()

Ejaan lokasi unik: 588 -> setelah title-case: 588
Dimensi one-hot (10 lokasi + Lainnya): 11 kolom


,lok_Bekasi,lok_Depok,lok_Kabupaten Bandung,lok_Kabupaten Bekasi,lok_Kabupaten Bogor,lok_Kabupaten Sidoarjo,lok_Kabupaten Tangerang,lok_Lainnya,lok_Semarang,lok_Surabaya,lok_Tangerang Selatan
0,0,0,0,0,0,0,0,1,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,0


## Langkah 7 — Standardisasi

Model berbasis jarak (kNN, SVM, KMeans, HDBSCAN) dan JST sangat sensitif terhadap skala.
`StandardScaler` mengubah setiap fitur menjadi rata-rata 0 dan simpangan baku 1.

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FITUR_NUM])

print('Rata-rata setelah scaling :', np.round(X_scaled.mean(axis=0), 6))
print('Std setelah scaling       :', np.round(X_scaled.std(axis=0), 6))
print('Shape fitur ternormalisasi:', X_scaled.shape)

Rata-rata setelah scaling : [0. 0. 0. 0.]
Std setelah scaling       : [1. 1. 1. 1.]
Shape fitur ternormalisasi: (56170, 4)


## Langkah 8 — Eksplorasi Setelah Praproses

Tiga hal yang dicek: hubungan fitur numerik dengan harga, sebaran harga per kondisi,
dan keseimbangan kelas.

In [8]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=df, x='kondisi', y='harga', order=['Baru', 'Bekas', 'Tidak diketahui'],
            ax=ax[0], showfliers=False)
ax[0].set_yscale('log')
ax[0].set_title('Harga per kondisi')
ax[0].set_xlabel('')

sns.boxplot(data=df, x='kelas_harga', y='luas_m2',
            order=['Murah', 'Sedang', 'Mahal'], ax=ax[1], showfliers=False)
ax[1].set_title('Luas tanah per kelas harga')

sns.scatterplot(data=df.sample(3000, random_state=42), x='luas_m2', y='harga',
                hue='kelas_harga', hue_order=['Murah', 'Sedang', 'Mahal'],
                alpha=0.5, s=15, ax=ax[2])
ax[2].set_yscale('log')
ax[2].set_title('Luas tanah vs harga (3.000 sampel)')

plt.tight_layout()
plt.show()

In [9]:
plt.figure(figsize=(7, 5))
corr = df[FITUR_NUM + ['harga']].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Korelasi fitur numerik dengan harga')
plt.tight_layout()
plt.show()

**Analisis JS02:**

- Pembersihan membuang **8.480 baris (13,1%)**: 7.156 duplikat persis (hasil scraping beberapa portal) dan 1.324 baris dengan luas/harga/kamar tidak wajar. Setelah itu tersisa **56.170 baris**.
- Distribusi kelas hampir sempurna: **18.723 Murah / 18.727 Sedang / 18.720 Mahal**, sehingga akurasi bisa dibandingkan langsung dengan *baseline* 1/3 ≈ 0,333.
- `usia_bangunan` hanya terisi pada **7,7% baris** (2.757 nilai tahun dikonversi), sehingga `kondisi` didominasi `Tidak diketahui` (51.838 baris; Bekas 3.415; Baru 917). Fitur ini disimpan untuk kelengkapan, tetapi kontribusinya kecil.
- Korelasi dengan harga: **`luas_bangunan` paling kuat (r = 0,637)**, lalu `luas_m2` (0,514), `jumlah_kamar_mandi` (0,398), dan `jumlah_kamar` (0,335). Ini masuk akal: harga properti terutama ditentukan luas bangunan/tanah dan lokasi.
- `luas_m2` dan `luas_bangunan` berkorelasi 0,686, sedangkan `jumlah_kamar` dan `jumlah_kamar_mandi` 0,758. Korelasi ini belum sampai menimbulkan multikolinearitas serius (VIF < 3, lihat JS03), tetapi tetap perlu diperhatikan pada model linear.
- `lokasi` memiliki **588 ejaan berbeda** setelah title-case (dari 595 ejaan mentah), sehingga normalisasi/encoding menjadi langkah wajib sebelum pemodelan.
- Standardisasi menghasilkan rata-rata 0 dan simpangan baku 1 untuk keempat fitur numerik, siap dipakai model berbasis jarak.
